# Task 3 — ML Model Portfolio

**Module:** 7006SCN Machine Learning and Big Data  
**Student:** VARSHAREDDY RAJIREDDYGARI

In [ ]:
# ── Step 1: Install PySpark ──────────────────────────────────────────────────
!pip install pyspark -q

In [ ]:
# ── Step 2: Download dataset if not already present ─────────────────────────
import urllib.request, os

DATA_URL  = "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2024-01.parquet"
DATA_PATH = "/content/fhvhv_tripdata_2024-01.parquet"

if not os.path.exists(DATA_PATH):
    print("Downloading NYC TLC HVFHV dataset (~451 MB) …")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Download complete.")
else:
    print(f"Dataset already present at {DATA_PATH}")

print(f"File size: {os.path.getsize(DATA_PATH)/1e6:.1f} MB")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, hour, dayofweek
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.classification import (LogisticRegression, DecisionTreeClassifier,
                                        RandomForestClassifier, GBTClassifier)
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import time, os, pandas as pd

spark = SparkSession.builder \
    .appName("7006SCN_Task3") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

## 3.1 Build Processed Data (self-contained — no dependency on Task 2 output)

In [ ]:
# Rebuild the processed dataset directly from the raw file
df_raw = spark.read.parquet(DATA_PATH)
df_raw = df_raw.withColumn("is_tipped", when(col("tips") > 0, 1).otherwise(0))
df_sampled = df_raw.sample(withReplacement=False, fraction=0.005, seed=42)
df_clean = df_sampled.dropna(
    subset=["trip_miles", "trip_time", "base_passenger_fare", "PULocationID", "DOLocationID"]
)
df_features = df_clean \
    .withColumn("pickup_hour", hour(col("pickup_datetime"))) \
    .withColumn("pickup_day",  dayofweek(col("pickup_datetime")))

selected_cols = ["PULocationID", "DOLocationID", "trip_miles", "trip_time",
                 "base_passenger_fare", "pickup_hour", "pickup_day", "is_tipped"]
df_final = df_features.select(selected_cols)

categorical_cols = ["PULocationID", "DOLocationID", "pickup_hour", "pickup_day"]
numeric_cols     = ["trip_miles", "trip_time", "base_passenger_fare"]
stages = []
for c in categorical_cols:
    stages += [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep"),
               OneHotEncoder(inputCols=[c+"_idx"], outputCols=[c+"_vec"])]
stages += [VectorAssembler(inputCols=numeric_cols, outputCol="num_raw"),
           StandardScaler(inputCol="num_raw", outputCol="num_scaled", withStd=True, withMean=True),
           VectorAssembler(inputCols=[c+"_vec" for c in categorical_cols]+["num_scaled"], outputCol="features")]

df_processed = Pipeline(stages=stages).fit(df_final).transform(df_final).select("features","is_tipped")
df_train, df_test = df_processed.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {df_train.count():,} | Test: {df_test.count():,}")

## 3.2 Evaluators

In [ ]:
ev_roc  = BinaryClassificationEvaluator(labelCol="is_tipped", metricName="areaUnderROC")
ev_acc  = MulticlassClassificationEvaluator(labelCol="is_tipped", predictionCol="prediction", metricName="accuracy")
ev_f1   = MulticlassClassificationEvaluator(labelCol="is_tipped", predictionCol="prediction", metricName="f1")
ev_prec = MulticlassClassificationEvaluator(labelCol="is_tipped", predictionCol="prediction", metricName="weightedPrecision")
ev_rec  = MulticlassClassificationEvaluator(labelCol="is_tipped", predictionCol="prediction", metricName="weightedRecall")

results = []

def run_cv(estimator, param_grid, name):
    cv = CrossValidator(estimator=estimator, estimatorParamMaps=param_grid,
                        evaluator=ev_roc, numFolds=3, seed=42)
    t0 = time.time()
    model = cv.fit(df_train)
    elapsed = time.time() - t0
    preds = model.transform(df_test)
    row = {
        "Model": name,
        "Accuracy":  round(ev_acc.evaluate(preds),  4),
        "Precision": round(ev_prec.evaluate(preds), 4),
        "Recall":    round(ev_rec.evaluate(preds),  4),
        "F1-Score":  round(ev_f1.evaluate(preds),   4),
        "AUC-ROC":   round(ev_roc.evaluate(preds),  4),
        "Train_Time_s": round(elapsed, 1)
    }
    results.append(row)
    print(f"  {name:30s} | Acc={row['Accuracy']} | F1={row['F1-Score']} | AUC={row['AUC-ROC']} | {elapsed:.1f}s")
    return model

## 3.3 Model 1 — Logistic Regression

In [ ]:
lr = LogisticRegression(labelCol="is_tipped", featuresCol="features")
run_cv(lr, ParamGridBuilder().addGrid(lr.regParam,[0.01,0.1]).addGrid(lr.elasticNetParam,[0.0,0.5]).build(), "Logistic Regression")

## 3.4 Model 2 — Decision Tree

In [ ]:
dt = DecisionTreeClassifier(labelCol="is_tipped", featuresCol="features")
run_cv(dt, ParamGridBuilder().addGrid(dt.maxDepth,[5,10]).build(), "Decision Tree")

## 3.5 Model 3 — Random Forest

In [ ]:
rf = RandomForestClassifier(labelCol="is_tipped", featuresCol="features")
run_cv(rf, ParamGridBuilder().addGrid(rf.numTrees,[10,20]).addGrid(rf.maxDepth,[5]).build(), "Random Forest")

## 3.6 Model 4 — Gradient-Boosted Trees

In [ ]:
gbt = GBTClassifier(labelCol="is_tipped", featuresCol="features")
run_cv(gbt, ParamGridBuilder().addGrid(gbt.maxDepth,[5]).addGrid(gbt.maxIter,[10]).build(), "Gradient-Boosted Trees")

## 3.7 Model Summary Table

In [ ]:
os.makedirs("/content/outputs", exist_ok=True)
results_df = pd.DataFrame(results)
results_df.to_csv("/content/outputs/model_metrics.csv", index=False)
results_df